# Preprocessing pipeline review (Google Colab / Drive)

Top-to-bottom review of **NIfTI / DICOM → nnUNet-ready labels**, focused on:

1. **Background vs `other-tissue`** quality
2. **Stable visualisation** (same organ → same colour)
3. Whether **bad inputs** explain low GTVp Dice (vs nnUNet itself)

Audit set: **4 RADCURE** + **4 HECKTOR** cases.

| nnUNet stem | Original ID (used for download) |
|-------------|-------------------------------|
| `case_0122` | `RADCURE-0122` |
| `case_0040` | `RADCURE-0040` |
| `case_0397` | `RADCURE-0397` |
| `case_0151` | `RADCURE-0151` |
| `case_012` | `HMR-012` *(provisional center; falls back to any matching number)* |
| `case_023` | `CHUM-023` *(provisional)* |
| `case_098` | `CHUM-098` *(provisional)* |
| `case_057` | `HMR-057` *(provisional)* |

**Colab tip:** after the install cell → **Runtime → Restart session** → remount Drive → run the import-only cell (skip pip).


## 0. Colab setup — Drive, repo, package


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# Paths: prefer an existing clone on Drive; otherwise clone from GitHub.
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/phD")
WORK_DIR = DRIVE_ROOT / "preprocessing_pipeline_review"
WORK_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_REPO = DRIVE_ROOT / "repos" / "radcure-medical-imaging"
GITHUB_REPO_URL = "https://github.com/xiscapericas/my_tailors_drawer.git"
CLONE_DIR = Path("/content/my_tailors_drawer")
REPO_SUBPATH = Path("psyduck-doing-phd/radcure-medical-imaging")

if DRIVE_REPO.is_dir() and (DRIVE_REPO / "setup.py").is_file():
    REPO_ROOT = DRIVE_REPO
    print("Using Drive repo:", REPO_ROOT)
else:
    if not (CLONE_DIR / REPO_SUBPATH / "setup.py").is_file():
        !git clone --depth 1 {GITHUB_REPO_URL} {CLONE_DIR}
    REPO_ROOT = CLONE_DIR / REPO_SUBPATH
    print("Using cloned repo:", REPO_ROOT)

assert (REPO_ROOT / "setup.py").is_file(), f"setup.py not found under {REPO_ROOT}"
assert (REPO_ROOT / "image_processor").is_dir()
os.chdir(REPO_ROOT)
print("cwd:", Path.cwd())


In [ ]:
# Install — Step A deps only (NO TotalSegmentator yet).
# MUST install from REPO_ROOT (not /content).
# After this cell: Runtime → Restart session → remount Drive → import-only cell.

import os
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/phD")
DRIVE_REPO = DRIVE_ROOT / "repos" / "radcure-medical-imaging"
CLONE_DIR = Path("/content/my_tailors_drawer")
REPO_SUBPATH = Path("psyduck-doing-phd/radcure-medical-imaging")

if DRIVE_REPO.is_dir() and (DRIVE_REPO / "setup.py").is_file():
    REPO_ROOT = DRIVE_REPO
elif (CLONE_DIR / REPO_SUBPATH / "setup.py").is_file():
    REPO_ROOT = CLONE_DIR / REPO_SUBPATH
else:
    raise FileNotFoundError(
        "Cannot find radcure-medical-imaging (setup.py missing). "
        "Re-run Drive mount + clone/path cell first."
    )

os.chdir(REPO_ROOT)
print("Installing from:", REPO_ROOT)

pkgs = [
    "numpy==2.0.2",
    "boto3",
    "python-dotenv",
    "blosc2>=2.5.0",
    "nibabel",
    "SimpleITK",
    "pydicom",
    "rt-utils",
    "matplotlib",
    "scikit-image",
    "scipy",
    "opencv-python-headless",
    "tqdm",
    "p-tqdm",
    "seaborn",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs],
    cwd=str(REPO_ROOT),
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)],
    cwd=str(REPO_ROOT),
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy==2.0.2"],
    cwd=str(REPO_ROOT),
)

print("Install done from", REPO_ROOT)
print(">>> Runtime → Restart session")
print(">>> Then: remount Drive → run NEXT cell (skip this pip cell)")


In [ ]:
# Run AFTER Runtime → Restart session.
# Re-run Drive mount if needed, then this cell. Skip pip.

import os
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/phD")
WORK_DIR = DRIVE_ROOT / "preprocessing_pipeline_review"
WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_REPO = DRIVE_ROOT / "repos" / "radcure-medical-imaging"
CLONE_DIR = Path("/content/my_tailors_drawer")
REPO_SUBPATH = Path("psyduck-doing-phd/radcure-medical-imaging")

if DRIVE_REPO.is_dir() and (DRIVE_REPO / "setup.py").is_file():
    REPO_ROOT = DRIVE_REPO
else:
    REPO_ROOT = CLONE_DIR / REPO_SUBPATH

assert (REPO_ROOT / "image_processor").is_dir(), f"Repo not found at {REPO_ROOT}"
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import image_processor

print("cwd:", Path.cwd())
print("numpy:", np.__version__)
print("image_processor OK:", image_processor.__file__)


### Optional later — TotalSegmentator (only when we reach Step C)

Do **not** install yet. When needed:

```python
!pip install -q totalsegmentator
# Runtime → Restart session, then re-run the import-only cell
```


## 1. Config & AWS credentials

Do **not** hardcode secrets. Prefer Colab Secrets or `getpass`.

Required: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_BUCKET_NAME`, `AWS_FOLDER`.
Optional: `HECKTOR_S3_URI`.


In [ ]:
import os
from getpass import getpass
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/phD")
WORK_DIR = DRIVE_ROOT / "preprocessing_pipeline_review"
WORK_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata

    def _secret(name: str, default: str = "") -> str:
        try:
            return userdata.get(name)
        except Exception:
            return default
except ImportError:

    def _secret(name: str, default: str = "") -> str:
        return default


def ensure_env(key: str, prompt: str, secret: bool = False) -> str:
    val = os.environ.get(key) or _secret(key)
    if not val:
        val = getpass(prompt) if secret else input(prompt)
    os.environ[key] = val.strip()
    return os.environ[key]


ensure_env("AWS_ACCESS_KEY_ID", "AWS_ACCESS_KEY_ID: ", secret=True)
ensure_env("AWS_SECRET_ACCESS_KEY", "AWS_SECRET_ACCESS_KEY: ", secret=True)
os.environ.setdefault("AWS_REGION", _secret("AWS_REGION", "eu-west-1") or "eu-west-1")
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

AWS_BUCKET_NAME = ensure_env(
    "AWS_BUCKET_NAME", "AWS_BUCKET_NAME (e.g. xisca-lab): ", secret=False
)
AWS_FOLDER = (
    os.environ.get("AWS_FOLDER")
    or _secret("AWS_FOLDER", "RADCURE/all_cases/")
    or "RADCURE/all_cases/"
)
os.environ["AWS_FOLDER"] = AWS_FOLDER

HECKTOR_S3_URI = (
    os.environ.get("HECKTOR_S3_URI")
    or _secret("HECKTOR_S3_URI", "s3://xisca-lab/HECKTOR/test1.zip")
    or "s3://xisca-lab/HECKTOR/test1.zip"
)
os.environ["HECKTOR_S3_URI"] = HECKTOR_S3_URI

DATA_ROOT = WORK_DIR / "audit_cases"
RADCURE_ROOT = DATA_ROOT / "radcure"
HECKTOR_ROOT = DATA_ROOT / "hecktor"
HECKTOR_DOWNLOAD_DIR = DATA_ROOT / "hecktor_download"
ORGAN_DICT_PATH = WORK_DIR / "audit_organ_dictionary.json"

for p in (RADCURE_ROOT, HECKTOR_ROOT, HECKTOR_DOWNLOAD_DIR):
    p.mkdir(parents=True, exist_ok=True)

# --- Audit cases ---
# RADCURE: case_XXXX -> RADCURE-XXXX
RADCURE_CASE_IDS = ["RADCURE-0122", "RADCURE-0040", "RADCURE-0397", "RADCURE-0151"]

# HECKTOR stems case_012/023/098/057 — center provisional (seed 42).
# Download falls back to any matching *-NNN if preferred ID is missing.
HECKTOR_NNUNET_STEMS = ["case_012", "case_023", "case_098", "case_057"]
HECKTOR_CASE_IDS = ["HMR-012", "CHUM-023", "CHUM-098", "HMR-057"]

SLICE_EXPANSION = 5

print("DATA_ROOT:", DATA_ROOT)
print("RADCURE:", RADCURE_CASE_IDS)
print("HECKTOR stems:", HECKTOR_NNUNET_STEMS)
print("HECKTOR provisional IDs:", HECKTOR_CASE_IDS)
print("AWS:", AWS_BUCKET_NAME, AWS_FOLDER)
print("HECKTOR S3:", HECKTOR_S3_URI)


## 2. Download audit cases

### 2a. RADCURE — 4 case zips from S3

Uses `AWSHandler` + `FileHandler`. Does **not** run TotalSegmentator yet.


In [ ]:
from image_processor.io.aws_handler import AWSHandler
from image_processor.io.file_handler import FileHandler

aws = AWSHandler(
    bucket_name=AWS_BUCKET_NAME,
    aws_folder=AWS_FOLDER,
    region_name=os.environ["AWS_REGION"],
)

radcure_local = {}
for case_id in RADCURE_CASE_IDS:
    case_dir = RADCURE_ROOT / case_id
    zip_path = RADCURE_ROOT / f"{case_id}.zip"
    if not case_dir.is_dir():
        if not zip_path.is_file():
            print(f"Downloading {case_id} ...")
            aws.download_case(case_id, str(RADCURE_ROOT))
        print(f"Unzipping {case_id} ...")
        FileHandler.unzip_file(str(zip_path), str(case_dir))
    else:
        print(f"Already present: {case_dir}")
    radcure_local[case_id] = case_dir

print("RADCURE ready:", {k: str(v) for k, v in radcure_local.items()})


### 2b. HECKTOR — 4 cases

Priority:
1. Reuse under `HECKTOR_ROOT` if ready.
2. Else resolve IDs (exact or any matching number) from Drive or S3 zip, extract only those cases.


In [ ]:
import shutil
import zipfile
from urllib.parse import urlparse

from image_processor.conventions import get_hecktor_paths


def resolve_hecktor_case_id(preferred_id: str, available_ids: list) -> str:
    # Prefer exact ID; else any folder whose numeric suffix matches.
    if preferred_id in available_ids:
        return preferred_id
    num = preferred_id.split("-")[-1]
    matches = [a for a in available_ids if a.split("-")[-1] == num]
    if not matches:
        raise FileNotFoundError(
            f"No HECKTOR case for preferred {preferred_id} (suffix {num}). "
            f"Available sample: {available_ids[:15]}"
        )
    chosen = sorted(matches)[0]
    print(f"  {preferred_id} not found -> using {chosen} (suffix {num})")
    return chosen


def list_hecktor_case_ids_in_zip(zip_path: Path) -> list:
    ids = set()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for n in zf.namelist():
            for part in Path(n).parts:
                if part.endswith(".nii.gz") or part in ("", ".", "__MACOSX"):
                    continue
                if "-" in part:
                    tail = part.split("-")[-1]
                    if tail.isdigit() and 1 <= len(tail) <= 4:
                        ids.add(part)
    return sorted(ids)


def hecktor_case_ready(cases_root: Path, case_id: str) -> bool:
    paths = get_hecktor_paths(str(cases_root / case_id), case_id)
    return os.path.isfile(paths["path_ct"]) and os.path.isfile(paths["path_mask"])


def download_s3_file(s3_uri: str, local_path: Path, region: str = "eu-west-1") -> Path:
    import boto3

    parsed = urlparse(s3_uri)
    bucket, key = parsed.netloc, parsed.path.lstrip("/")
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.is_file():
        print("Zip already on disk:", local_path)
        return local_path
    print(f"Downloading {s3_uri} -> {local_path}")
    boto3.client("s3", region_name=region).download_file(bucket, key, str(local_path))
    return local_path


def extract_hecktor_cases_from_zip(zip_path: Path, case_ids: list, dest_root: Path) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        for case_id in case_ids:
            members = [
                n
                for n in names
                if f"/{case_id}/" in f"/{n}" or n.startswith(f"{case_id}/")
            ]
            if not members:
                raise FileNotFoundError(f"{case_id} not found inside {zip_path.name}")
            print(f"Extracting {case_id}: {len(members)} files")
            for m in members:
                zf.extract(m, dest_root)

    from pipelines.hecktor.test_pipeline import detect_hecktor_cases_root

    return Path(detect_hecktor_cases_root(str(dest_root)))


DRIVE_HECKTOR_CASES = DRIVE_ROOT / "phD-Petia" / "Trainings" / "Hecktor" / "cases"
HECKTOR_CASES_ROOT = None

# 1) Already mirrored under HECKTOR_ROOT
if all(hecktor_case_ready(HECKTOR_ROOT, c) for c in HECKTOR_CASE_IDS):
    HECKTOR_CASES_ROOT = HECKTOR_ROOT
    print("HECKTOR cases already under", HECKTOR_CASES_ROOT)

# 2) Drive folder: resolve centers by numeric suffix if needed
elif DRIVE_HECKTOR_CASES.is_dir():
    drive_ids = sorted(
        d.name
        for d in DRIVE_HECKTOR_CASES.iterdir()
        if d.is_dir() and not d.name.startswith(".")
    )
    print(f"Drive HECKTOR folders: n={len(drive_ids)}")
    try:
        resolved = [resolve_hecktor_case_id(c, drive_ids) for c in HECKTOR_CASE_IDS]
        if all(hecktor_case_ready(DRIVE_HECKTOR_CASES, c) for c in resolved):
            HECKTOR_CASE_IDS = resolved
            HECKTOR_CASES_ROOT = DRIVE_HECKTOR_CASES
            print("Using Drive HECKTOR cases:", HECKTOR_CASE_IDS)
    except FileNotFoundError as e:
        print("Drive resolve incomplete:", e)

# 3) S3 zip — extract only resolved cases
if HECKTOR_CASES_ROOT is None:
    zip_name = Path(urlparse(HECKTOR_S3_URI).path).name or "hecktor.zip"
    zip_path = download_s3_file(
        HECKTOR_S3_URI, HECKTOR_DOWNLOAD_DIR / zip_name, os.environ["AWS_REGION"]
    )
    available = list_hecktor_case_ids_in_zip(zip_path)
    print(f"HECKTOR IDs in zip: n={len(available)}")
    HECKTOR_CASE_IDS = [resolve_hecktor_case_id(c, available) for c in HECKTOR_CASE_IDS]
    print("Resolved HECKTOR_CASE_IDS:", HECKTOR_CASE_IDS)
    unpack_parent = HECKTOR_DOWNLOAD_DIR / "unzipped_partial"
    found_root = extract_hecktor_cases_from_zip(zip_path, HECKTOR_CASE_IDS, unpack_parent)
    for case_id in HECKTOR_CASE_IDS:
        src = found_root / case_id
        dst = HECKTOR_ROOT / case_id
        if src.is_dir() and not dst.exists():
            shutil.copytree(src, dst)
    HECKTOR_CASES_ROOT = HECKTOR_ROOT
    print("HECKTOR cases ready at", HECKTOR_CASES_ROOT)

for case_id in HECKTOR_CASE_IDS:
    assert hecktor_case_ready(HECKTOR_CASES_ROOT, case_id), f"Missing HECKTOR files for {case_id}"
print("All HECKTOR audit cases OK:", HECKTOR_CASE_IDS)


## 3. Case inventory

Confirm each case resolves to the expected inputs before any mask logic.


In [ ]:
from image_processor.io.file_handler import FileHandler
from image_processor.conventions import get_hecktor_paths

inventory = {"radcure": {}, "hecktor": {}}

for case_id, case_dir in radcure_local.items():
    dicom_folder = FileHandler.get_dicom_path(str(case_dir), case_id)
    paths = FileHandler.get_ct_and_mask_paths(dicom_folder)
    inventory["radcure"][case_id] = {
        "case_dir": str(case_dir),
        "dicom_folder": dicom_folder,
        "ct_path": paths["ct_path"],
        "rtstruct_path": paths["mask_path"],
    }
    print(f"[RADCURE] {case_id}")
    print("  CT:", paths["ct_path"])
    print("  RTSTRUCT:", paths["mask_path"])

for case_id in HECKTOR_CASE_IDS:
    paths = get_hecktor_paths(str(HECKTOR_CASES_ROOT / case_id), case_id)
    inventory["hecktor"][case_id] = paths
    print(f"[HECKTOR] {case_id}")
    print("  CT:", paths["path_ct"])
    print("  Mask:", paths["path_mask"])

print("\nInventory:", {k: list(v) for k, v in inventory.items()})


---

## 4. Pipeline walkthrough (top → bottom)

Same order as `CaseProcessor`, inspect after each stage:

```
RADCURE: DICOM+RTSTRUCT ─┐
                         ├─► CT volume + tumor mask
HECKTOR: NIfTI+mask ─────┘
         ├─► slice crop (tumor ± SLICE_EXPANSION)
         ├─► TotalSegmentator (organs)
         ├─► background / anatomical_region   ← suspect #1
         ├─► organs + leftover → other-tissue ← suspect #1
         ├─► overlay GTVp (/ GTVn)
         ├─► save case_*_0000.nii.gz
         └─► visualisation PDF                ← suspect #2
```


### Step A — Load CT + tumor (no TotalSegmentator yet)

RADCURE: DICOM → NIfTI, then RTSTRUCT **GTVp=1 / GTVn=2** via `load_labeled_tumor_volume`, then **`save_and_align_mask_with_ct`** (same as production — without this, tumor does not sit on the CT).

HECKTOR: load CT + mask NIfTI (already 1=GTVp, 2=GTVn).

Visualisation: **all** selected slices with separate GTVp (red) and GTVn (magenta).


In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from image_processor.io.nifti_handler import NIfTIHandler
from image_processor.core.dicom_handler import DICOMHandler
from image_processor.utils.image_processing import ImageProcessor

nifti_handler = NIfTIHandler()
dicom_handler = DICOMHandler()

loaded = {"radcure": {}, "hecktor": {}}


def tumor_slice_range(mask_vol: np.ndarray, expansion: int) -> list:
    non_zero = ImageProcessor.get_non_zero_slices(mask_vol)
    z = mask_vol.shape[2]
    if not non_zero:
        return list(range(z))
    start = max(int(min(non_zero)) - expansion, 0)
    end = min(int(max(non_zero)) + expansion, z - 1)
    return list(range(start, end + 1))


def summarize_tumor(tumor: np.ndarray) -> str:
    n_p = int(np.sum(tumor == 1))
    n_n = int(np.sum(tumor == 2))
    return f"labels={np.unique(tumor)}, GTVp_voxels={n_p}, GTVn_voxels={n_n}"


# --- RADCURE: convert CT, load separate GTVp/GTVn, ALIGN to CT (required) ---
for case_id, meta in inventory["radcure"].items():
    case_dir = meta["case_dir"]
    ct_nii = nifti_handler.convert_dicom_to_nifti(meta["ct_path"], case_id, case_dir)

    # Separate labels: 1=GTVp, 2=GTVn (GTVn skipped if absent in RTSTRUCT)
    tumor_raw = dicom_handler.load_labeled_tumor_volume(
        meta["ct_path"], meta["rtstruct_path"]
    )
    print(f"[RADCURE] {case_id}: raw RTSTRUCT {summarize_tumor(tumor_raw)} shape={tumor_raw.shape}")

    aligned_path = str(Path(case_dir) / f"{case_id}_tumor_mask_aligned.nii.gz")
    nifti_handler.save_and_align_mask_with_ct(tumor_raw, ct_nii, aligned_path)

    ct = nib.load(ct_nii).get_fdata().astype(np.float32)
    tumor = nib.load(aligned_path).get_fdata().astype(np.int32)
    if tumor.shape != ct.shape:
        raise ValueError(
            f"{case_id}: aligned tumor {tumor.shape} != CT {ct.shape}"
        )

    slices = tumor_slice_range(tumor, SLICE_EXPANSION)
    loaded["radcure"][case_id] = {
        "ct_path": ct_nii,
        "tumor_path": aligned_path,
        "ct": ct,
        "tumor": tumor,
        "slices": slices,
    }
    print(
        f"[RADCURE] {case_id}: aligned CT {ct.shape}, {summarize_tumor(tumor)}, "
        f"slices {slices[0]}..{slices[-1]} (n={len(slices)})"
    )

# --- HECKTOR: already separate GTVp/GTVn on disk ---
for case_id, paths in inventory["hecktor"].items():
    ct = nib.load(paths["path_ct"]).get_fdata().astype(np.float32)
    tumor = nib.load(paths["path_mask"]).get_fdata().astype(np.int32)
    slices = tumor_slice_range(tumor, SLICE_EXPANSION)
    loaded["hecktor"][case_id] = {
        "ct_path": paths["path_ct"],
        "mask_path": paths["path_mask"],
        "ct": ct,
        "tumor": tumor,
        "slices": slices,
    }
    print(
        f"[HECKTOR] {case_id}: CT {ct.shape}, {summarize_tumor(tumor)}, "
        f"slices {slices[0]}..{slices[-1]} (n={len(slices)})"
    )


In [ ]:
def show_ct_tumor_all_slices(
    case_id: str,
    ct: np.ndarray,
    tumor: np.ndarray,
    slices: list,
    title_prefix: str,
    ncols: int = 4,
):
    """All selected slices: CT + separate GTVp (red) / GTVn (magenta)."""
    n = len(slices)
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))

    crop = ct[:, :, slices]
    p1, p99 = np.percentile(crop, (1, 99))
    denom = (p99 - p1) + 1e-8

    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax_i, idx in enumerate(slices):
        ax = axes[ax_i]
        ct_s = np.clip((ct[:, :, idx] - p1) / denom, 0, 1)
        m = tumor[:, :, idx]
        ax.imshow(ct_s.T, cmap="gray", origin="lower")
        # Keep GTVp and GTVn separate (do not merge)
        if np.any(m == 1):
            ax.imshow(
                np.ma.masked_where(m != 1, np.ones_like(m)).T,
                cmap="Reds",
                alpha=0.55,
                origin="lower",
                vmin=0,
                vmax=1,
            )
        if np.any(m == 2):
            ax.imshow(
                np.ma.masked_where(m != 2, np.ones_like(m)).T,
                cmap="spring",
                alpha=0.55,
                origin="lower",
                vmin=0,
                vmax=1,
            )
        n_p = int(np.sum(m == 1))
        n_n = int(np.sum(m == 2))
        ax.set_title(f"z={idx} p={n_p} n={n_n}", fontsize=8)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(
        f"{title_prefix} {case_id} — {n} slices | GTVp=red GTVn=magenta",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


for case_id, d in loaded["radcure"].items():
    print(f"RADCURE {case_id}: plotting {len(d['slices'])} slices")
    show_ct_tumor_all_slices(case_id, d["ct"], d["tumor"], d["slices"], "RADCURE")

for case_id, d in loaded["hecktor"].items():
    print(f"HECKTOR {case_id}: plotting {len(d['slices'])} slices")
    show_ct_tumor_all_slices(case_id, d["ct"], d["tumor"], d["slices"], "HECKTOR")


### Step A2 — Anatomy QC (auto-discard non-human / broken FOV)

Heuristic **human-anatomy likelihood** in `[0, 1]` from:
- tumor presence (GTVp preferred; GTVn-only weaker)
- CT intensity dynamic range
- patient fill fraction (`head_mask_from_array` on sample slices)
- body-mask coherence (largest connected component)
- number of selected slices

Cases with `score < ANATOMY_QC_THRESHOLD` are **discarded** from `loaded` for later steps.
All decisions are logged (JSONL); discards also get a CSV summary.


In [ ]:
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

from image_processor.utils.anatomy_qc import (
    score_human_anatomy,
    apply_anatomy_threshold,
    append_qc_log,
    write_discard_summary_csv,
)

# Tune after inspecting scores on the audit set
ANATOMY_QC_THRESHOLD = 0.55

QC_DIR = WORK_DIR / "logs" / "anatomy_qc"
QC_DIR.mkdir(parents=True, exist_ok=True)
QC_JSONL = QC_DIR / "anatomy_qc_decisions.jsonl"
QC_DISCARD_CSV = QC_DIR / "anatomy_qc_discarded.csv"

# Fresh run: optional wipe of previous JSONL for this session
# QC_JSONL.write_text("")

qc_rows = []
discard_rows = []
kept = {"radcure": {}, "hecktor": {}}

for convention in ("radcure", "hecktor"):
    for case_id, d in loaded[convention].items():
        result = score_human_anatomy(d["ct"], d["tumor"], d["slices"])
        keep, record = apply_anatomy_threshold(result, threshold=ANATOMY_QC_THRESHOLD)
        record["case_id"] = case_id
        record["convention"] = convention
        append_qc_log(str(QC_JSONL), case_id=case_id, convention=convention, record=record)
        qc_rows.append(record)

        comps = record["components"]
        print(
            f"[{convention}] {case_id}: score={record['score']:.3f} "
            f"→ {record['decision']} | "
            f"tumor={comps['tumor']:.2f} fill={comps['patient_fill']:.2f} "
            f"coh={comps['body_coherence']:.2f} inten={comps['intensity']:.2f}"
        )
        if record["reasons"]:
            print(f"    reasons: {record['reasons']}")

        if keep:
            kept[convention][case_id] = d
        else:
            discard_rows.append(record)

# Restrict downstream work to kept cases
loaded = kept
n_keep = sum(len(v) for v in loaded.values())
n_drop = len(discard_rows)
print(f"\nKept {n_keep} cases, discarded {n_drop}")
print("Log:", QC_JSONL)

if discard_rows:
    write_discard_summary_csv(discard_rows, str(QC_DISCARD_CSV))
    print("Discard CSV:", QC_DISCARD_CSV)
else:
    print("No discards at threshold", ANATOMY_QC_THRESHOLD)

# Quick table
summary = []
# Re-read decisions from this run
for r in qc_rows:
    summary.append({
        "convention": r["convention"],
        "case_id": r["case_id"],
        "score": round(r["score"], 3),
        "decision": r["decision"],
        "gtvp": r["metrics"].get("gtvp_voxels"),
        "fill": round(r["metrics"].get("mean_patient_fill", 0), 3),
        "reasons": ";".join(r.get("reasons") or []),
    })
if pd is not None:
    display(pd.DataFrame(summary).sort_values(["decision", "score"]))
else:
    for row in sorted(summary, key=lambda r: (r["decision"], r["score"])):
        print(row)


### Step B — Background / anatomical_region *(next — main suspect)*

Not run yet. Next we call `head_mask_from_array` / `generate_background_array` and inspect patient vs air/table and `other-tissue`.

**Do not skip ahead** until Step A overlays look correct (tumor aligned with CT).


### Later steps (placeholders)

| Step | Topic | Status |
|------|--------|--------|
| C | TotalSegmentator organs | pending |
| D | Combined mask + `other-tissue` | pending |
| E | Tumor overlay | pending |
| F | Save nnUNet NIfTIs | pending |
| G | Stable colormap visualisation | pending |
| H | DatasetXXX / QA gates | pending |

Notes:

- Step A findings:
- Step B findings:
- ...
